In [40]:
# ============================================================
# МОДУЛЬ 4: PRODUCTION SYSTEM - КОМПЛЕКСНЫЙ ИНФЕРЕНС (v2.1 - FIXED)
# ============================================================
# 
# ИСПРАВЛЕНИЯ V2.1:
#   ✅ Фиксена ошибка индексирования normalized_weights (numpy array)
#   ✅ Добавлена загрузка fraud_keywords из JSON в __init__
#   ✅ Фиксена обработка transcription (берётся  элемент)
#   ✅ Все веса правильно индексируются в detailed_analysis
#
# АРХИТЕКТУРА:
#   Audio Input
#       ↓
#   ├─→ Module 1 (WavLM + AASIST) ──→ score_deepfake
#   ├─→ Module 2 (Whisper + Prosody) ──→ score_keyword + score_prosody
#   └─→ Module 3 (Fusion - LogisticRegression) ──→ Final Decision

import torch
import torch.nn as nn
from transformers import AutoModel, WhisperProcessor, WhisperForConditionalGeneration
import numpy as np
import librosa
import os
import pickle
import json
from scipy import signal
from pathlib import Path
import warnings
from fuzzywuzzy import fuzz
warnings.filterwarnings('ignore')

print("=" * 90)
print("🚀 ИНИЦИАЛИЗАЦИЯ PRODUCTION SYSTEM - VOICE FRAUD DETECTION (v2.1)")
print("=" * 90)

# ============================================================
# ЧАСТЬ 1: МОДУЛЬ 1 - DEEPFAKE DETECTION (WAVLM + AASIST)
# ============================================================

class DeepfakeInferenceModule:
    """
    Инференсный модуль для обнаружения синтезированной речи (deepfake).
    
    Архитектура:
    - WavLM-large: извлечение признаков из сырого аудио
    - AAiST Classifier: классификация на bonafide/spoofed
    """
    
    def __init__(self, 
                 checkpoint_path: str,
                 device: str = "cuda"):
        
        self.device = torch.device(device)
        self.sr = 16000
        self.target_duration = 3.0
        self.target_samples = int(self.sr * self.target_duration)  # 48000
        
        print("\n📦 МОДУЛЬ 1: DeepfakeInferenceModule")
        print("-" * 90)
        print("🔧 Инициализация...")
        
        # 1️⃣ ЗАГРУЗИТЬ WAVLM
        print(" 📥 WavLM-large...", end=" ")
        self.wavlm = AutoModel.from_pretrained("microsoft/wavlm-large")
        self.wavlm.eval()
        for param in self.wavlm.parameters():
            param.requires_grad = False
        self.wavlm = self.wavlm.to(self.device)
        self.wavlm_hidden_dim = self.wavlm.config.hidden_size  # 1024
        print("✅")
        
        # 2️⃣ СОЗДАТЬ И ЗАГРУЗИТЬ AASIST
        print(" 🏗️  AAiST Classifier...", end=" ")
        self.aasist = self._create_aasist_model()
        
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"❌ Checkpoint не найден: {checkpoint_path}")
        
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.aasist.load_state_dict(checkpoint['model_state_dict'])
        self.aasist.eval()
        print("✅")
        
        print("✅ Module 1 готов к инференсу")
    
    def _create_aasist_model(self):
        """Создаёт AAiST модель"""
        
        class AAiSTClassifier(nn.Module):
            def __init__(self, input_dim: int = 1024, hidden_dim: int = 256):
                super().__init__()
                
                self.encoder = nn.Sequential(
                    nn.Linear(input_dim, hidden_dim),
                    nn.ReLU(),
                    nn.BatchNorm1d(hidden_dim),
                    nn.Dropout(0.2)
                )
                
                self.classifier = nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim // 2),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(hidden_dim // 2, 2)
                )
            
            def forward(self, x):
                x = self.encoder(x)
                logits = self.classifier(x)
                return logits
        
        model = AAiSTClassifier(input_dim=self.wavlm_hidden_dim)
        model = model.to(self.device)
        return model
    
    def _load_audio(self, audio_path: str) -> np.ndarray:
        """Загружает и нормализует аудио"""
        try:
            y, sr = librosa.load(audio_path, sr=self.sr, mono=True)
            
            # Нормализация
            max_val = np.max(np.abs(y))
            if max_val > 0:
                y = y / (max_val + 1e-7)
            
            # Фильтр высоких частот (убрать шум)
            sos = signal.butter(5, 200, 'hp', fs=self.sr, output='sos')
            y = signal.sosfilt(sos, y)
            
            # Pad/Trim до целевой длины
            if len(y) > self.target_samples:
                y = y[:self.target_samples]
            else:
                y = np.pad(y, (0, self.target_samples - len(y)), mode='constant')
            
            return y
        
        except Exception as e:
            print(f"  ⚠️  Ошибка загрузки аудио: {e}")
            return None
    
    def get_score(self, audio_path: str) -> float:
        """
        Получает вероятность deepfake для одного файла.
        
        Args:
            audio_path: путь к аудиофайлу
        
        Returns:
            float: вероятность что это deepfake [0, 1]
        """
        try:
            # Загрузить аудио
            y = self._load_audio(audio_path)
            if y is None:
                return 0.5
            
            # Преобразовать в тензор
            audio_tensor = torch.FloatTensor(y).unsqueeze(0).to(self.device)  # [1, 48000]
            
            # Извлечь признаки (WavLM)
            with torch.no_grad():
                wavlm_embeddings = self.wavlm(audio_tensor)
                # Global Average Pooling по временной оси
                wavlm_pooled = wavlm_embeddings.last_hidden_state.mean(dim=1)  # [1, 1024]
            
            # Классификация (AAiST)
            with torch.no_grad():
                logits = self.aasist(wavlm_pooled)  # [1, 2]
                probs = torch.softmax(logits, dim=1)  # [1, 2]
            
            # Вероятность deepfake (класс 1)
            prob_deepfake = float(probs[0, 1].cpu().numpy())
            
            return prob_deepfake
        
        except Exception as e:
            print(f"  ⚠️  Ошибка при инференсе Module 1: {e}")
            return 0.5


# ============================================================
# ЧАСТЬ 2: МОДУЛЬ 2 - FRAUD PATTERN DETECTION (WHISPER + PROSODY)
# ============================================================

class FraudPatternInferenceModule:
    """
    Инференсный модуль для обнаружения мошеннических паттернов.
    
    Два компонента:
    1. Whisper: транскрипция речи и детекция ключевых фраз
    2. Prosody: анализ просодических характеристик (F0, duration, etc)
    """
    
    def __init__(self, 
                 whisper_model_path: str = None,
                 prosody_model_path: str = None,
                 fraud_keywords_path: str = None,
                 device: str = "cuda"):
        
        self.device = torch.device(device)
        self.sr = 16000
        
        print("\n📦 МОДУЛЬ 2: FraudPatternInferenceModule")
        print("-" * 90)
        print("🔧 Инициализация...")
        
        # 1️⃣ ЗАГРУЗИТЬ WHISPER
        print(" 📥 Whisper (base)...", end=" ")
        self.processor = WhisperProcessor.from_pretrained("openai/whisper-base")
        self.whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")
        self.whisper_model.eval()
        for param in self.whisper_model.parameters():
            param.requires_grad = False
        self.whisper_model = self.whisper_model.to(self.device)
        print("✅")
        
        # 2️⃣ ЗАГРУЗИТЬ PROSODY MODEL
        print(" 🎵 Prosody Model (RF)...", end=" ")
        if prosody_model_path and os.path.exists(prosody_model_path):
            with open(prosody_model_path, 'rb') as f:
                self.prosody_model = pickle.load(f)
            print("✅")
        else:
            print("⚠️  (используется stub)")
            self.prosody_model = None
        
        # 3️⃣ ЗАГРУЗИТЬ МОШЕННИЧЕСКИЕ ФРАЗЫ ✅ ИСПРАВЛЕНО
        print(" 📋 Fraud Keywords...", end=" ")
        self.fraud_keywords = self._load_fraud_keywords(fraud_keywords_path)
        print(f"✅ ({len(self.fraud_keywords)} фраз)")
        
        print("✅ Module 2 готов к инференсу")
    
    def _load_fraud_keywords(self, json_path: str = None) -> list:
        """
        Загружает мошеннические фразы из JSON файла.
        
        Если JSON не найден, используется fallback список.
        """
        
        # Fallback список (на случай если JSON не найден)
        fallback_keywords = [
            'срочно', 'перевод', 'банк', 'счёт', 'пароль', 'пин',
            'подтверждение', 'деньги', 'платёж', 'карта', 'кредит',
            'срочная', 'ответ', 'быстро', 'позвоните', 'прямо',
            'код', 'номер', 'сообщение', 'дом', 'прямых', 'звоните'
        ]
        
        # Попытка загрузить из JSON
        if json_path and os.path.exists(json_path):
            try:
                with open(json_path, 'r', encoding='utf-8') as f:
                    phrases_dict = json.load(f)
                
                # Извлечь только ключи (фразы)
                keywords = list(phrases_dict.keys())
                print(f"\n  ✅ Загружено {len(keywords)} фраз из JSON")
                return keywords
            
            except Exception as e:
                print(f"\n  ⚠️  Ошибка загрузки JSON: {e}")
                print(f"  📌 Используется fallback список ({len(fallback_keywords)} фраз)")
                return fallback_keywords
        else:
            if json_path:
                print(f"\n  ⚠️  JSON не найден: {json_path}")
            print(f"  📌 Используется fallback список ({len(fallback_keywords)} фраз)")
            return fallback_keywords
    
    def _load_audio(self, audio_path: str) -> np.ndarray:
        """Загружает аудио"""
        try:
            y, sr = librosa.load(audio_path, sr=self.sr, mono=True)
            
            # Нормализация
            max_val = np.max(np.abs(y))
            if max_val > 0:
                y = y / (max_val + 1e-7)
            
            return y
        except Exception as e:
            print(f"  ⚠️  Ошибка загрузки аудио: {e}")
            return None
    
    def _extract_keywords_score(self, text: str) -> float:
        """
        Извлекает score ключевых слов.
        
        Нормализованное число обнаруженных мошеннических фраз [0, 1]
        """
        transcript = text.lower()
         # transcript = "вас беспокоит служба безопасности сбербанка, замечена подозрительная операция по вашему счету, назовите код"
            
           # Исправляем типичные ошибки Whisper
        corrections = {
            'нод': 'код',
            'нодов': 'кодов',
            'сэм': 'смс',
            'сем': 'смс',
            'кредиа': 'кредита',
            'эфб': 'фсб',
        }
        for wrong, correct in corrections.items():
            transcript = transcript.replace(wrong, correct)

        print("B")
        print(transcript)
        print("B")
            
            # Ищем мошеннические фразы
        keyword_score = 0
        found = []
            
        for phrase in self.fraud_keywords:
                # Точное совпадение
            if phrase in transcript:
                keyword_score += 0.5
                print("НАШЕЛ")
                found.append(phrase)
                # Все слова есть в тексте (не требует порядка)
            elif len(phrase.split()) > 1 and all(w in transcript for w in phrase.split()):
                keyword_score += 0.3
                print("НАШЕЛ 2")
                # Нечёткое совпадение с НИЗКИМ порогом (65 вместо 85)
            elif fuzz.token_set_ratio(phrase, transcript) > 65:
                keyword_score += 0.2
                print("НАШЕЛ 3")
            
            # Нормализуем
        keyword_score = min(keyword_score / 10, 1.0)  # Делим на 10
        print("A")
        print(keyword_score)
        print("A")
                
        return keyword_score
    
    def _extract_prosody_features(self, audio: np.ndarray) -> float:
        """
        Извлекает просодические признаки и возвращает вероятность мошенничества.
        
        Returns:
            float: вероятность мошеннической просодии [0, 1]
        """
        try:
            if self.prosody_model is None:
                # Если модель не загружена, используем простой heuristic
                return 0.3
            
            # Используй свою реальную функцию из Module2
            return 0.3  # Placeholder
        
        except Exception as e:
            print(f"  ⚠️  Ошибка при извлечении просодии: {e}")
            return 0.3
    
    def get_scores(self, audio_path: str) -> tuple:
        """
        Получает scores для одного аудиофайла.
        
        Returns:
            (keyword_score, prosody_score): оба в диапазоне [0, 1]
        """
        try:
            # Загрузить аудио
            audio = self._load_audio(audio_path)
            if audio is None:
                return 0.5, 0.5
            
            # 1️⃣ WHISPER ТРАНСКРИПЦИЯ
            inputs = self.processor(audio, sampling_rate=self.sr, return_tensors="pt")
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                generated_ids = self.whisper_model.generate(
                    inputs["input_features"],
                    task="transcribe",
                    language="<|ru|>"  # Измени на <|ru|> если русский
                )
            
            transcription = str(self.processor.batch_decode(
                generated_ids,
                skip_special_tokens=True
            )[0])  # ✅ ИСПРАВЛЕНО: берём  элемент списка
            
            # 2️⃣ EXTRACT KEYWORD SCORE
            keyword_score = self._extract_keywords_score(transcription)
            
            # 3️⃣ EXTRACT PROSODY SCORE
            prosody_score = self._extract_prosody_features(audio)
            
            return keyword_score, prosody_score
        
        except Exception as e:
            print(f"  ⚠️  Ошибка при инференсе Module 2: {e}")
            return 0.5, 0.5


# ============================================================
# ЧАСТЬ 3: МОДУЛЬ 3 - DECISION FUSION (ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ)
# ============================================================

class FusionDecisionModule:
    """
    Модуль объединения решений через обученную Логистическую Регрессию.
    
    Входы:
    - score_deepfake: вероятность от Module 1
    - score_keyword: оценка ключевых слов от Module 2
    - score_prosody: оценка просодии от Module 2
    
    Выходы:
    - final_score: финальная вероятность мошенничества [0, 1]
    - decision: '🟢 NORM' или '🔴 FRAUD'
    - confidence: уверенность в решении [0, 1]
    """
    
    def __init__(self, 
                 fusion_model_path: str,
                 threshold: float = None):
        
        print("\n📦 МОДУЛЬ 3: FusionDecisionModule")
        print("-" * 90)
        print("🔧 Инициализация...")
        
        # Загрузить fusion модель
        print(" 📥 Fusion Model (LogisticRegression)...", end=" ")
        
        if not os.path.exists(fusion_model_path):
            raise FileNotFoundError(f"❌ Fusion модель не найдена: {fusion_model_path}")
        
        with open(fusion_model_path, 'rb') as f:
            self.fusion_package = pickle.load(f)
        
        self.model = self.fusion_package['model']
        self.scaler = self.fusion_package['scaler']
        self.normalized_weights = self.fusion_package['normalized_weights']
        self.feature_names = self.fusion_package['feature_names']
        self.metrics = self.fusion_package['metrics']
        self.statistics = self.fusion_package['statistics']
        
        # Threshold
        if threshold is None:
            self.threshold = self.fusion_package.get('optimal_threshold', 0.5)
        else:
            self.threshold = threshold
        
        print("✅")
        
        print(f"\n⚖️  Нормализованные веса решений:")
        print(f"   🔊 Deepfake (WavLM):  {self.normalized_weights[0]*100:.1f}%")
        print(f"   📝 Keywords (Whisper): {self.normalized_weights[1]*100:.1f}%")
        print(f"   🎵 Prosody:           {self.normalized_weights[2]*100:.1f}%")
        
        print(f"\n🎚️  Threshold: {self.threshold:.3f}")
        print(f"✅ Module 3 готов к инференсу")
    
    def make_decision(self,
                      score_deepfake: float,
                      score_keyword: float,
                      score_prosody: float) -> dict:
        """
        Делает финальное решение через Fusion.
        
        Args:
            score_deepfake: [0, 1]
            score_keyword: [0, 1]
            score_prosody: [0, 1]
        
        Returns:
            dict с final_score, decision, confidence
        """
        
        # Подготовить входные признаки
        X = np.array([[score_deepfake, score_keyword, score_prosody]])
        
        # Стандартизировать
        X_scaled = self.scaler.transform(X)
        
        # Получить вероятность через Logistic Regression
        final_score = float(self.model.predict_proba(X_scaled)[0, 1])
        
        # Статистическое решение
        if final_score > self.threshold:
            decision = '🔴 FRAUD'
            confidence = final_score
        else:
            decision = '🟢 NORM'
            confidence = 1.0 - final_score
        
        return {
            'final_score': final_score,
            'decision': decision,
            'confidence': confidence
        }



# ============================================================
# ЧАСТЬ 4: КОМПЛЕКСНАЯ PRODUCTION СИСТЕМА
# ============================================================

class VoiceFraudDetectionSystem:
    """
    Полная система для обнаружения голосового мошенничества.
    
    Интегрирует Module 1, 2 и 3 в единую pipeline.
    """
    
    def __init__(self,
                 module1_checkpoint: str,
                 module2_prosody_model: str = None,
                 module2_fraud_keywords: str = None,
                 module3_fusion_model: str = None,
                 threshold: float = None,
                 device: str = "cuda"):
        
        print("\n" + "=" * 90)
        print("🚀 ИНИЦИАЛИЗАЦИЯ ПОЛНОЙ СИСТЕМЫ")
        print("=" * 90)
        
        # Инициализировать все модули
        self.module1 = DeepfakeInferenceModule(
            checkpoint_path=module1_checkpoint,
            device=device
        )
        
        self.module2 = FraudPatternInferenceModule(
            prosody_model_path=module2_prosody_model,
            fraud_keywords_path=module2_fraud_keywords,  # ✅ ПЕРЕДАЁМ fraud_keywords_path
            device=device
        )
        
        self.module3 = FusionDecisionModule(
            fusion_model_path=module3_fusion_model,
            threshold=threshold
        )
        
        print("\n" + "=" * 90)
        print("✅ СИСТЕМА ПОЛНОСТЬЮ ИНИЦИАЛИЗИРОВАНА И ГОТОВА К РАБОТЕ!")
        print("=" * 90)
    
    def detect(self,
               audio_path: str,
               threshold: float = None) -> dict:
        """
        Основной метод детекции.
        
        Args:
            audio_path: путь к аудиофайлу
            threshold: опционально переопределить threshold
        
        Returns:
            dict с полной информацией о детекции
        """
        
        import time
        start_time = time.time()
        
        print(f"\n🔍 ОБРАБОТКА ФАЙЛА: {Path(audio_path).name}")
        print("-" * 90)
        
        try:
            # MODULE 1: Deepfake Detection
            print(" [1/3] Module 1: Deepfake Detection...", end=" ", flush=True)
            score_deepfake = self.module1.get_score(audio_path)
            print(f"✅ {score_deepfake:.4f}")
            
            # MODULE 2: Fraud Pattern Detection
            print(" [2/3] Module 2: Fraud Pattern Detection...", end=" ", flush=True)
            score_keyword, score_prosody = self.module2.get_scores(audio_path)
            print(f"✅ keyword={score_keyword:.4f}, prosody={score_prosody:.4f}")
            
            # MODULE 3: Fusion Decision
            print(" [3/3] Module 3: Fusion Decision...", end=" ", flush=True)
            
            if threshold is not None:
                self.module3.threshold = threshold
            
            fusion_result = self.module3.make_decision(
                score_deepfake=score_deepfake,
                score_keyword=score_keyword,
                score_prosody=score_prosody
            )
            print(f"✅ {fusion_result['final_score']:.4f}")
            
            elapsed_time = time.time() - start_time
            
            # Собрать финальный результат
            result = {
                'file': Path(audio_path).name,
                'score_deepfake': round(score_deepfake, 4),
                'score_keyword': round(score_keyword, 4),
                'score_prosody': round(score_prosody, 4),
                'final_score': round(fusion_result['final_score'], 4),
                'decision': fusion_result['decision'],
                'confidence': round(fusion_result['confidence'], 4),
                'processing_time': round(elapsed_time, 2),
                'detailed_analysis': {
                    'deepfake_contribution': round(float(self.module3.normalized_weights[0]) * float(score_deepfake), 4),
                    'keyword_contribution': round(float(self.module3.normalized_weights[1]) * float(score_keyword), 4),
                    'prosody_contribution': round(float(self.module3.normalized_weights[2]) * float(score_prosody), 4),
                    'threshold_used': float(self.module3.threshold)
                }

            }
            
            return result
        
        except Exception as e:
            print(f"❌ ОШИБКА: {e}")
            return {
                'file': Path(audio_path).name,
                'error': str(e),
                'decision': '⚠️  ERROR'
            }
    
    def detect_batch(self,
                     audio_paths: list,
                     threshold: float = None) -> list:
        """
        Обработка нескольких файлов.
        
        Args:
            audio_paths: список путей к файлам
            threshold: опционально переопределить threshold
        
        Returns:
            list с результатами для каждого файла
        """
        
        results = []
        for i, audio_path in enumerate(audio_paths, 1):
            print(f"\n[{i}/{len(audio_paths)}]", end=" ")
            result = self.detect(audio_path, threshold=threshold)
            results.append(result)
        
        return results


# ============================================================
# ИНИЦИАЛИЗАЦИЯ СИСТЕМЫ
# ============================================================

if __name__ == "__main__":
    
    # ПУТИ К МОДЕЛЯМ И ДАННЫМ
    MODULE1_CHECKPOINT = "/kaggle/input/module1/pytorch/default/1/module1_aasist_best.pth"
    MODULE2_PROSODY_MODEL = "/kaggle/input/module2-prosodyc/scikitlearn/default/1/prosodic_model.pkl"
    MODULE2_FRAUD_KEYWORDS = "/kaggle/input/malicious-phrases/phrases.json"  # ✅ JSON PATH
    MODULE3_FUSION_MODEL = "/kaggle/input/decision/scikitlearn/default/1/fusion_model.pkl"
    
    # ИНИЦИАЛИЗИРОВАТЬ СИСТЕМУ
    system = VoiceFraudDetectionSystem(
        module1_checkpoint=MODULE1_CHECKPOINT,
        module2_prosody_model=MODULE2_PROSODY_MODEL,
        module2_fraud_keywords=MODULE2_FRAUD_KEYWORDS,  # ✅ ПЕРЕДАЁМ JSON PATH
        module3_fusion_model=MODULE3_FUSION_MODEL,
        threshold=None,
        device="cuda"
    )
    
    print("\n" + "=" * 90)
    print("✅ СИСТЕМА ГОТОВА!")
    print("=" * 90)
    print("""
Примеры использования:

1️⃣  Одиночный файл:
    result = system.detect("/path/to/audio.wav")
    print(result)

2️⃣  Несколько файлов:
    results = system.detect_batch([audio1, audio2, audio3])

3️⃣  С кастомным threshold:
    result = system.detect(audio, threshold=0.4)

4️⃣  Проверка результата:
    if result['decision'] == '🔴 FRAUD':
        print("⚠️  ОБНАРУЖЕНО МОШЕННИЧЕСТВО!")
    else:
        print("✅ Голос подлинный")
""")



🚀 ИНИЦИАЛИЗАЦИЯ PRODUCTION SYSTEM - VOICE FRAUD DETECTION (v2.1)

🚀 ИНИЦИАЛИЗАЦИЯ ПОЛНОЙ СИСТЕМЫ

📦 МОДУЛЬ 1: DeepfakeInferenceModule
------------------------------------------------------------------------------------------
🔧 Инициализация...
 📥 WavLM-large... ✅
 🏗️  AAiST Classifier... ✅
✅ Module 1 готов к инференсу

📦 МОДУЛЬ 2: FraudPatternInferenceModule
------------------------------------------------------------------------------------------
🔧 Инициализация...
 📥 Whisper (base)... ✅
 🎵 Prosody Model (RF)... ✅
 📋 Fraud Keywords... 
  ✅ Загружено 491 фраз из JSON
✅ (491 фраз)
✅ Module 2 готов к инференсу

📦 МОДУЛЬ 3: FusionDecisionModule
------------------------------------------------------------------------------------------
🔧 Инициализация...
 📥 Fusion Model (LogisticRegression)... ✅

⚖️  Нормализованные веса решений:
   🔊 Deepfake (WavLM):  74.9%
   📝 Keywords (Whisper): 2.7%
   🎵 Prosody:           22.4%

🎚️  Threshold: 0.081
✅ Module 3 готов к инференсу

✅ СИСТЕМА ПОЛНОСТЬЮ И

In [41]:
# В конце ноутбука (после основного кода):
system = VoiceFraudDetectionSystem(
    module1_checkpoint="/kaggle/input/module1/pytorch/default/1/module1_aasist_best.pth",
    module2_prosody_model="/kaggle/input/module2-prosodyc/scikitlearn/default/1/prosodic_model.pkl",
    module2_fraud_keywords="/kaggle/input/malicious-phrases/phrases.json",
    module3_fusion_model="/kaggle/input/decision/scikitlearn/default/1/fusion_model.pkl",
    device="cuda"
)



🚀 ИНИЦИАЛИЗАЦИЯ ПОЛНОЙ СИСТЕМЫ

📦 МОДУЛЬ 1: DeepfakeInferenceModule
------------------------------------------------------------------------------------------
🔧 Инициализация...
 📥 WavLM-large... ✅
 🏗️  AAiST Classifier... ✅
✅ Module 1 готов к инференсу

📦 МОДУЛЬ 2: FraudPatternInferenceModule
------------------------------------------------------------------------------------------
🔧 Инициализация...
 📥 Whisper (base)... ✅
 🎵 Prosody Model (RF)... ✅
 📋 Fraud Keywords... 
  ✅ Загружено 491 фраз из JSON
✅ (491 фраз)
✅ Module 2 готов к инференсу

📦 МОДУЛЬ 3: FusionDecisionModule
------------------------------------------------------------------------------------------
🔧 Инициализация...
 📥 Fusion Model (LogisticRegression)... ✅

⚖️  Нормализованные веса решений:
   🔊 Deepfake (WavLM):  74.9%
   📝 Keywords (Whisper): 2.7%
   🎵 Prosody:           22.4%

🎚️  Threshold: 0.081
✅ Module 3 готов к инференсу

✅ СИСТЕМА ПОЛНОСТЬЮ ИНИЦИАЛИЗИРОВАНА И ГОТОВА К РАБОТЕ!


In [44]:
# Одиночный файл
result = system.detect("/kaggle/input/testdata/2025-12-24-22.57.35.wav")
print(result)

# Или несколько файлов
# results = system.detect_batch([file1, file2, file3])



🔍 ОБРАБОТКА ФАЙЛА: 2025-12-24-22.57.35.wav
------------------------------------------------------------------------------------------
 [1/3] Module 1: Deepfake Detection... ✅ 0.0000
 [2/3] Module 2: Fraud Pattern Detection... B
 добрый день, александр, у вас беспокоит службы безопасности т-банка. на вашем счете, зададоктировано подозрительная активность.
B
НАШЕЛ 3
НАШЕЛ 3
НАШЕЛ 3
A
0.06000000000000001
A
✅ keyword=0.0600, prosody=0.3000
 [3/3] Module 3: Fusion Decision... ✅ 0.0140
{'file': '2025-12-24-22.57.35.wav', 'score_deepfake': 0.0, 'score_keyword': 0.06, 'score_prosody': 0.3, 'final_score': 0.014, 'decision': '🟢 NORM', 'confidence': 0.986, 'processing_time': 0.55, 'detailed_analysis': {'deepfake_contribution': 0.0, 'keyword_contribution': 0.0016, 'prosody_contribution': 0.0673, 'threshold_used': 0.08080808080808081}}
